Adapted from LLaMa_1B_LoRA_with_evidence.ipynb

Added quantization option for CustomModel  
Modified TrainerModule to take a teacher and student model for distillation  
Commented out many stuff to prouduce preprocessed data and run scorer as I do that manually  
Changed paths to match my setup  


In [ ]:
!pip install -q transformers peft evaluate tomli scikit-learn pandas tqdm torch accelerate bitsandbytes
!pip install -U bitsandbytes accelerate

In [ ]:
# =========================
# 1. Imports
# =========================

import os
import re
import json
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
    BitsAndBytesConfig
)

from peft import LoraConfig, get_peft_model


In [ ]:
# =========================
# 2. Paths and config
# =========================

#os.chdir(r"C:\Users\aiger\Documents\2026SS\AIR\git_personal\AIR_Group_Task")
#print("Current directory:", os.getcwd())

#RAW_TRAIN_PATH = "data/english/english_train.json"
#VAL_PATH = "data/english/clef2026_gpt4_o_mini_val.json"

#TRAIN_JSONL = "output/training_data_for_RM/english_train_with_evidence.jsonl"

TRAIN_PATH = "../output/preprocessed_data/english_train_with_evidence.jsonl"
TEST_PATH = "../data/english/clef2026_gpt4_o_mini_val.json"


# Separate teacher output dirs
EXPERIMENT_NAME = "llama_evi_dist"
PRED_FILE_NAME = "clef_predictions.json"
TEACHER_MODEL_DIR = f"../output/{EXPERIMENT_NAME}/"
#TEACHER_PRED_PATH = f"../output/{EXPERIMENT_NAME}/RM_prediction/teacher_llama_evidence_predictions.json"
TEACHER_PRED_PATH = f"../output/{EXPERIMENT_NAME}/results/{PRED_FILE_NAME}"
TEACHER_RESULT_DIR = f"../output/{EXPERIMENT_NAME}/results/"

TEACHER_MODEL = "meta-llama/Llama-3.2-3B"
BASE_MODEL = "meta-llama/Llama-3.2-1B" # "meta-llama/Llama-3.2-1B"

MAX_LENGTH = 126 #256
BATCH_SIZE = 4 #2
EPOCHS = 5
LR = 1e-4
RANDOM_STATE = 42

#os.makedirs("output/training_data_for_RM", exist_ok=True)
#os.makedirs("output/RM_prediction", exist_ok=True)
os.makedirs(TEACHER_MODEL_DIR, exist_ok=True)
os.makedirs(TEACHER_RESULT_DIR, exist_ok=True)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

print("Device:", DEVICE)
print("Train file exists:", os.path.exists(TRAIN_PATH))
print("Val file exists:", os.path.exists(TEST_PATH))
#print("Preprocessing script exists:", os.path.exists("experiments/teacher_llama_evidence/reasoning_trace_build_with_evidence.py"))


In [ ]:
#torch.set_default_dtype(torch.float16)

In [ ]:
# =========================
# 4. Run evidence-aware preprocessing script

# subprocess.run(
#     [
#         sys.executable,
#         "experiments/teacher_llama_evidence/reasoning_trace_build_with_evidence.py",
#         "--input",
#         RAW_TRAIN_PATH,
#         "--output",
#         TRAIN_JSONL,
#     ],
#     check=True,
# )

train_df = pd.read_json(TRAIN_PATH, lines=True)
print(train_df["input_text"].iloc[0][:1000])
print(train_df.head())
print(train_df.columns)
print(train_df["Class"].value_counts())


In [ ]:
# =========================
# 5. Helper functions
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def get_evidence(sample):
    possible_keys = [
        "evidences",
        "evidence",
        "Evidence",
        "relevant_evidence",
        "Relevant_evidence",
        "context",
        "Context",
        "gold_evidence",
        "Gold_evidence",
    ]

    for key in possible_keys:
        if key in sample and sample[key]:
            value = sample[key]

            if isinstance(value, list):
                return " ".join(map(str, value))

            if isinstance(value, dict):
                return json.dumps(value, ensure_ascii=False)

            return str(value)

    return ""


def build_teacher_input(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Evidence: {evidence}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_params} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )



In [ ]:
# =========================
# 6. Use evidence-aware training text
# =========================

print("Training examples:", len(train_df))
print("Columns:", train_df.columns)

train_df["teacher_input_text"] = train_df["input_text"]

print(train_df[["teacher_input_text", "Class"]].head())

In [ ]:
# =========================
# 7. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["teacher_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item



In [ ]:
# =========================
# 8. LLaMA teacher verifier model
# =========================

# previous CustomClassifier, i guess
class CustomModel(torch.nn.Module):
    def __init__(
            self,
            model_name,
            is_teacher=False,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
            use_quant=False
    ):
        super().__init__()#

        if use_quant:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16
            )
            self.model = AutoModel.from_pretrained(TEACHER_MODEL, quantization_config=bnb_config,)
        else:
            self.model = AutoModel.from_pretrained(model_name)

        if is_teacher:
            self.model.eval()  # freeze teacher
            for p in self.model.parameters():
                p.requires_grad = False

        if use_lora:
            #lora_config = LoraConfig(
            #    r=lora_rank,
            #    lora_alpha=lora_alpha,
            #    target_modules=["q_proj", "k_proj", "v_proj"],
            #    lora_dropout=0.05,
            #    bias="none",
            #)
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules = [
                    "q_proj", "k_proj", "v_proj", "o_proj",
                    #"gate_proj", "up_proj", "down_proj"
                ],
                lora_dropout=dropout_value,
                bias="lora_only",
            )

            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, return_hidden=False):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # For decoder-style models: use last token representation
        pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)

        if return_hidden:
            return logits, pooled_output
        else:
            return logits



In [ ]:
# =========================
# 9. Trainer
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            teacher,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
            alpha=0.1,
            temperature=2.0,
    ):
        self.device = DEVICE #torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.teacher = teacher.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.alpha = alpha
        self.temperature = temperature

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        self.scaler = torch.amp.GradScaler('cuda')

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def distillation_loss(self, student_logits, teacher_logits, T=2.0):
        student_probs = torch.sigmoid(student_logits / T)
        teacher_probs = torch.sigmoid(teacher_logits / T)
    
        #return F.mse_loss(student_probs, teacher_probs)
        return F.binary_cross_entropy(student_probs, teacher_probs)
        
    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                with torch.autocast(device_type="cuda", dtype=torch.float16):

                    input_ids = batch["input_ids"].to(self.device)
                    attention_mask = batch["attention_mask"].to(self.device)
                    labels = batch["labels"].to(self.device)
                   
                    student_logits = self.model(input_ids, attention_mask)
                    student_logits = student_logits.float()

                    with torch.no_grad():
                        teacher_logits = self.teacher(input_ids, attention_mask)
                        teacher_logits = teacher_logits.float()

                    ce_loss = self.loss_fn(
                        student_logits.squeeze(-1),  # (B, 1) → (B,)
                        labels.float()
                    )
                    
                    #kd_loss = self.distillation_loss(student_logits, teacher_logits, self.temperature)
                    kd_loss = F.mse_loss(student_logits, teacher_logits)

                    #loss_hidden = F.mse_loss(student_hidden, teacher_hidden.detach())
                    
                    alpha = 0.7
                    beta = 0.3
                    #gamma = 0.4
                    loss = (alpha * ce_loss + beta * kd_loss) #+ gamma * loss_hidden
                    #loss.backward()
          
                    #self.optimizer.step()
                    #self.scheduler.step()

                    total_loss += loss.item()
    
                    preds = (
                            torch.sigmoid(student_logits).squeeze(1) >= 0.5
                    ).detach().cpu().numpy()
    
                    total_acc += accuracy_score(
                        labels.detach().cpu().numpy(),
                        preds,
                    )
                
                # critical change
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    input_ids = batch["input_ids"].to(self.device)
                    attention_mask = batch["attention_mask"].to(self.device)
                    labels = batch["labels"].to(self.device)
    
                    logits = self.model(input_ids, attention_mask)
                    loss = self.loss_fn(logits.squeeze(1), labels)
    
                    total_loss += loss.item()
    
                    preds = (
                            torch.sigmoid(logits).squeeze(1) >= 0.5
                    ).detach().cpu().numpy()
    
                    total_acc += accuracy_score(
                        labels.detach().cpu().numpy(),
                        preds,
                    )

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"teacher_model_epoch_{epoch}.pt"),
        )


In [ ]:
# =========================
# 10. Train teacher verifier
# =========================

train_split, dev_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=RANDOM_STATE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
dev_dataset = TextDataset(dev_split, tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
)

student_model = CustomModel(
    model_name=BASE_MODEL,
    use_lora=True,
    lora_rank=16,
    lora_alpha=32,
)

teacher_model = CustomModel(
    model_name=TEACHER_MODEL,
    is_teacher = True,
    use_lora=False,
    lora_rank=8,
    lora_alpha=16,
    use_quant=True
)

print_trainable_parameters(student_model)

trainer = TrainerModule(
    model=student_model,
    teacher=teacher_model,
    train_loader=train_loader,
    val_loader=dev_loader,
    epochs=EPOCHS,
    lr=LR,
    output_dir=TEACHER_MODEL_DIR,
)

trainer.train()


In [ ]:
# =========================
# 11. Teacher evaluator
# =========================

class TeacherEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
            device="cuda",
    ):
        self.device = DEVICE #torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomModel(
            model_name=base_model,
            use_lora=True,
            lora_rank=16,
            lora_alpha=32,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def encode_input(
            self,
            claim,
            evidence,
            verdict,
            justification,
            max_length=MAX_LENGTH,
    ):
        text = build_teacher_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidence, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            score = self.model(input_ids, attention_mask).item()

        return float(score)


In [ ]:
# =========================
# 12. Generate teacher predictions
# =========================

BEST_EPOCH = EPOCHS - 1

TEACHER_MODEL_PATH = os.path.join(
    TEACHER_MODEL_DIR,
    f"teacher_model_epoch_{BEST_EPOCH}.pt",
)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

teacher_evaluator = TeacherEvaluator(
    model_path=TEACHER_MODEL_PATH,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

predictions = []

for idx, sample in enumerate(tqdm(val_data)):
    claim = sample["claim"]
    evidence = get_evidence(sample)

    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx in range(len(sample["Reasoning_traces"])):
        justification = remove_label_pattern(
            sample["Reasoning_traces"][trace_idx]
        ).split("Label:")[0]

        verdict = sample["Verdict_list"][trace_idx].lower()

        score = teacher_evaluator.score(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        verdict_list.append(sample["Verdict_list"][trace_idx])
        justification_list.append(justification)
        verifier_score_list.append(score)

    best_idx = int(np.argmax(np.array(verifier_score_list)))
    best_verdict = verdict_list[best_idx]

    predictions.append(
        {
            "query_id": sample.get("query_id", idx),
            "Claim": claim,
            "Evidence": evidence,
            "Label": sample["label"],
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list": verifier_score_list,
        }
    )

with open(TEACHER_PRED_PATH, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved teacher predictions to {TEACHER_PRED_PATH}")
print("Number of predictions:", len(predictions))


In [ ]:
# =========================
# 13. Run provided scorer
# =========================

# The scorer expects this exact path:
# output/RM_prediction/clef_predictions.json

# shutil.copy(
#     TEACHER_PRED_PATH,
#     "output/RM_prediction/clef_predictions.json",
# )

# subprocess.run(
#     [sys.executable, "task2/scorer.py"],
#     check=True,
# )

# shutil.copy(
#     "output/RM_prediction/result.csv",
#     f"{TEACHER_RESULT_DIR}/result.csv",
# )

# shutil.copy(
#     "output/RM_prediction/per_sample_ir.csv",
#     f"{TEACHER_RESULT_DIR}/per_sample_ir.csv",
# )

# print(f"Teacher result saved to {TEACHER_RESULT_DIR}/result.csv")
# print(f"Teacher per-sample IR saved to {TEACHER_RESULT_DIR}/per_sample_ir.csv")